In [1]:
# Configuração
from pathlib import Path
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/AMMCI_PRF")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)

Projeto: d:\Workspace\ws-uem\acidentes-rodoviarios-ml


# 07 — Diário de experimentos

## Objetivo

Registrar as decisões que levaram ao modelo final. Não há treinamento neste notebook.

Repetições com seeds diferentes pertencem ao mesmo experimento.

In [2]:
# Cada linha representa uma decisão metodológica, não uma repetição por seed.
diary = pd.DataFrame([
    {
        "Experimento": "E01",
        "Notebook": "03",
        "Alteração": "Softmax manual com learning rate = 0.03",
        "Hipótese": "Uma taxa menor poderia tornar a otimização mais estável.",
        "Resultado": "Macro-F1 = 0.2908",
        "Interpretação": "O modelo permaneceu próximo do comportamento concentrado na classe majoritária; a taxa menor não trouxe aprendizagem útil no orçamento testado.",
        "Decisão": "Descartar."
    },
    {
        "Experimento": "E02",
        "Notebook": "03",
        "Alteração": "Softmax manual com 100 épocas",
        "Hipótese": "Um treinamento mais curto poderia ser suficiente.",
        "Resultado": "Macro-F1 = 0.2908",
        "Interpretação": "Cem épocas foram insuficientes para produzir bom equilíbrio entre as três classes.",
        "Decisão": "Descartar."
    },
    {
        "Experimento": "E03",
        "Notebook": "03",
        "Alteração": "Softmax manual sem pesos de classe",
        "Hipótese": "Servir como baseline da implementação própria.",
        "Resultado": "Accuracy = 0.7767; Macro-F1 = 0.3236",
        "Interpretação": "A acurácia alta não se converteu em desempenho equilibrado entre classes.",
        "Decisão": "Manter como baseline."
    },
    {
        "Experimento": "E04",
        "Notebook": "03",
        "Alteração": "Softmax manual com pesos balanceados",
        "Hipótese": "Pesos de classe poderiam melhorar o reconhecimento das classes minoritárias.",
        "Resultado": "Macro-F1 = 0.4755; Macro-Recall = 0.5962",
        "Interpretação": "O balanceamento aumentou fortemente Macro-F1 e Macro-Recall, apesar da redução de Accuracy.",
        "Decisão": "Manter como configuração principal da implementação manual."
    },
    {
        "Experimento": "E05",
        "Notebook": "03",
        "Alteração": "LogisticRegression do scikit-learn com balanceamento",
        "Hipótese": "A biblioteca deveria apresentar comportamento próximo da Softmax manual.",
        "Resultado": "Macro-F1 = 0.4790",
        "Interpretação": "O resultado ficou muito próximo da implementação própria, sustentando que a Softmax manual está aprendendo de forma coerente.",
        "Decisão": "Manter como comparação com biblioteca."
    },
    {
        "Experimento": "E06",
        "Notebook": "04",
        "Alteração": "MLP multi-hot sem pesos",
        "Hipótese": "Medir o desempenho da MLP sem tratamento de desbalanceamento.",
        "Resultado": "Macro-F1 = 0.4962 ± 0.0060; recall fatal = 0.1648",
        "Interpretação": "A rede obtém boa Accuracy, mas ainda reconhece pouco a classe fatal.",
        "Decisão": "Manter como baseline da MLP."
    },
    {
        "Experimento": "E07",
        "Notebook": "04",
        "Alteração": "MLP multi-hot com alpha = 0.50",
        "Hipótese": "Um balanceamento intermediário poderia melhorar as minoritárias sem penalizar demais a classe majoritária.",
        "Resultado": "Macro-F1 = 0.5495 ± 0.0015",
        "Interpretação": "O balanceamento intermediário melhorou bastante o equilíbrio e ficou próximo da melhor configuração.",
        "Decisão": "Manter como alternativa."
    },
    {
        "Experimento": "E08",
        "Notebook": "04",
        "Alteração": "MLP multi-hot com balanceamento completo",
        "Hipótese": "Pesos máximos poderiam maximizar a sensibilidade às classes minoritárias.",
        "Resultado": "Macro-F1 = 0.4837 ± 0.0231; recall fatal = 0.6091",
        "Interpretação": "O recall fatal aumentou, mas o desempenho global e a estabilidade pioraram.",
        "Decisão": "Não selecionar."
    },
    {
        "Experimento": "E09",
        "Notebook": "04",
        "Alteração": "MLP combinada: alpha = 0.50 + L2 + dropout",
        "Hipótese": "Balanceamento moderado e regularização poderiam produzir melhor compromisso global.",
        "Resultado": "Macro-F1 = 0.5574 ± 0.0024",
        "Interpretação": "Foi a maior média de Macro-F1 entre as configurações simplificadas da MLP.",
        "Decisão": "Selecionar como modelo principal."
    },
    {
        "Experimento": "E10",
        "Notebook": "05",
        "Alteração": "MFT-A: learning rate = 0.0001, até 20 épocas com early stopping",
        "Hipótese": "Uma atualização curta poderia adaptar M0 aos dados recentes sem alterar demais os pesos históricos.",
        "Resultado": "Macro-F1 em D1-val = 0.5530 ± 0.0038",
        "Interpretação": "Apresentou a maior média entre as duas configurações de fine-tuning.",
        "Decisão": "Selecionar para o MFT final."
    },
    {
        "Experimento": "E11",
        "Notebook": "05",
        "Alteração": "MFT-B: learning rate menor e treinamento mais longo",
        "Hipótese": "Uma atualização mais conservadora poderia melhorar a adaptação.",
        "Resultado": "Macro-F1 em D1-val = 0.5502 ± 0.0037",
        "Interpretação": "Ficou muito próxima de MFT-A, mas com média ligeiramente menor.",
        "Decisão": "Não selecionar."
    },
    {
        "Experimento": "E12",
        "Notebook": "05/06",
        "Alteração": "Comparação final M0 × MFT × MRT × MREC em D2",
        "Hipótese": "Comparar manutenção por fine-tuning, retreinamento completo e uso somente dos dados recentes.",
        "Resultado": "M0 = 0.5619; MFT = 0.5650; MRT = 0.5645; MREC = 0.5471 (Macro-F1 médio)",
        "Interpretação": "MFT e MRT ficaram praticamente empatados e levemente acima de M0; MREC ficou abaixo, indicando que o histórico continua útil.",
        "Decisão": "Usar MFT como principal estratégia de atualização, deixando claro que a vantagem é pequena."
    },
])

assert len(diary) >= 8

display(diary)

diary.to_csv(
    EXPERIMENT_DIR / "experiment_log.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"\nExperimentos registrados: {len(diary)}")
print("Diário salvo em:", EXPERIMENT_DIR / "experiment_log.csv")

,Experimento,Notebook,Alteração,Hipótese,Resultado,Interpretação,Decisão
0,E01,03,Softmax manual com learning rate = 0.03,Uma taxa menor poderia tornar a otimização mai...,Macro-F1 = 0.2908,O modelo permaneceu próximo do comportamento c...,Descartar.
1,E02,03,Softmax manual com 100 épocas,Um treinamento mais curto poderia ser suficiente.,Macro-F1 = 0.2908,Cem épocas foram insuficientes para produzir b...,Descartar.
2,E03,03,Softmax manual sem pesos de classe,Servir como baseline da implementação própria.,Accuracy = 0.7767; Macro-F1 = 0.3236,A acurácia alta não se converteu em desempenho...,Manter como baseline.
3,E04,03,Softmax manual com pesos balanceados,Pesos de classe poderiam melhorar o reconhecim...,Macro-F1 = 0.4755; Macro-Recall = 0.5962,O balanceamento aumentou fortemente Macro-F1 e...,Manter como configuração principal da implemen...
4,E05,03,LogisticRegression do scikit-learn com balance...,A biblioteca deveria apresentar comportamento ...,Macro-F1 = 0.4790,O resultado ficou muito próximo da implementaç...,Manter como comparação com biblioteca.
5,E06,04,MLP multi-hot sem pesos,Medir o desempenho da MLP sem tratamento de de...,Macro-F1 = 0.4962 ± 0.0060; recall fatal = 0.1648,"A rede obtém boa Accuracy, mas ainda reconhece...",Manter como baseline da MLP.
6,E07,04,MLP multi-hot com alpha = 0.50,Um balanceamento intermediário poderia melhora...,Macro-F1 = 0.5495 ± 0.0015,O balanceamento intermediário melhorou bastant...,Manter como alternativa.
7,E08,04,MLP multi-hot com balanceamento completo,Pesos máximos poderiam maximizar a sensibilida...,Macro-F1 = 0.4837 ± 0.0231; recall fatal = 0.6091,"O recall fatal aumentou, mas o desempenho glob...",Não selecionar.
8,E09,04,MLP combinada: alpha = 0.50 + L2 + dropout,Balanceamento moderado e regularização poderia...,Macro-F1 = 0.5574 ± 0.0024,Foi a maior média de Macro-F1 entre as configu...,Selecionar como modelo principal.
9,E10,05,"MFT-A: learning rate = 0.0001, até 20 épocas c...",Uma atualização curta poderia adaptar M0 aos d...,Macro-F1 em D1-val = 0.5530 ± 0.0038,Apresentou a maior média entre as duas configu...,Selecionar para o MFT final.



Experimentos registrados: 12
Diário salvo em: d:\Workspace\ws-uem\acidentes-rodoviarios-ml\outputs\experiments\experiment_log.csv


## Síntese

Principais conclusões:

1. o balanceamento melhorou Softmax e MLP;
2. a MLP combinada foi selecionada em D0;
3. MFT e MRT ficaram próximos; MREC teve pior resultado.

O ganho de MFT sobre M0 é pequeno e não comprova concept drift.